In [64]:
import ROOT as rt
import ctypes
%jsroot on
file = rt.TFile("e21062_44S.root")
dtime = file.Get("dtime")

In [124]:

def decay_daughter(dim, par):
    """
    Function for fitting decay curves
    a = amplitude of source
    b = tau (decay constant) of parent
    c = tau (decay constant) of daughter
    bkgrd = background
    Bateman equation
    """

    a = par[0]
    b = par[1]
    c = par[2]
    d = par[3]
    bkgrd = par[4]
    x = dim[0]
    return d*((b*a)/(c-b))*(rt.TMath.Exp(-b*x) - rt.TMath.Exp(-c*x)) + bkgrd

def Decayp(hist):
    
    MaxX = hist.GetXaxis().GetXmax();
    MinX = hist.GetXaxis().GetXmin();
    # Axis range
    MinX = 0
    #MaxX = 100
    
    print("Min X: ", MinX)
    print("Max X: ", MaxX)
    
    # Canvas
    fitgraph = rt.TCanvas("fits", "fits", 800, 600)
    fitgraph.Clear()

    # Parent half-life guess
    Parent_halflife = rt.TMath.Log(2) / 122.8
    Daughter_halflife = rt.TMath.Log(2) / (542 + 1)

    # Define TF1
    decayp = rt.TF1("decayp", decay_daughter, MinX, MaxX, 5)

    decayp.SetParName(0, "source #")
    decayp.SetParName(1, "Source lambda")
    decayp.SetParName(2, "Daughter lambda")
    decayp.SetParName(3, "scaling factor")
    decayp.SetParName(4, "shift")

    decayp.SetParameters(600, Parent_halflife, Daughter_halflife, .3,400) # intial conditions

    decayp.SetParLimits(0, 400, 2000) # source
    decayp.SetParLimits(1, rt.TMath.Log(2)/(122.8 + 3), rt.TMath.Log(2)/(122.8 - 3)) # lambda
    decayp.SetParLimits(2, 1e-6, .004) # lambda
    decayp.SetParLimits(3, .01, 1) # scale
    decayp.SetParLimits(4, 0, 800) # bckgrnd

    # Print initial parameters
    for i in range(decayp.GetNpar()):
        print(f"Initial p{i} = {decayp.GetParameter(i)}")

    # Perform fit
    hist.Fit("decayp", "RS", "", MinX, MaxX)
    hist.Print("V")
    print("\n")
    
    # Print parameter limits
    for i in range(decayp.GetNpar()):
        par_min = ctypes.c_double()
        par_max = ctypes.c_double()
        decayp.GetParLimits(i, par_min, par_max)
        print(f"Parameter {i} ({decayp.GetParName(i)}) limits: [{par_min}, {par_max}]")

    # Half-life calculation
    lambda_fit = decayp.GetParameter(1)
    lambda_err = decayp.GetParError(1)

    lambda_daughter = decayp.GetParameter(2)
    lambda_daughter_err = decayp.GetParError(2) 

    decayhl = rt.TMath.Log(2) / lambda_fit
    decay_error = (rt.TMath.Log(2) / (lambda_fit**2)) * lambda_err

    decay_daughter_hl = rt.TMath.Log(2) / lambda_daughter
    decay_daughter_error = (rt.TMath.Log(2) / (lambda_daughter**2)) * lambda_daughter_err

    print(f"Decay: {decayhl:.6f} ms ± {decay_error:.6f} ms")
    print(f"Decay Daughter: {decay_daughter_hl:.6f} ms ± {decay_daughter_error:.6f} ms")

    # Chi2 / NDF
    chi2 = decayp.GetChisquare()
    ndf = decayp.GetNDF()
    print(f"Chi^2/NDF: {chi2/ndf:.6f}\n")

    # Constant background function
    bckgrnd = rt.TF1("bckgrnd", "[0]", MinX, MaxX)
    bckgrnd.SetParameter(0, decayp.GetParameter(2))
    bckgrnd.SetLineColor(rt.kBlack)

    # Legend
    legend = rt.TLegend(0.7, 0.4, 0.9, 0.7)
    legend.AddEntry(hist, "Data", "E")
    legend.AddEntry(decayp, "Fit", "l")
    legend.AddEntry(bckgrnd, "backgrnd", "l")

    # Styling
    hist.SetTitle("Decay of 44Ar Gated on 1157 keV;Time (ms);Counts")
    hist.GetXaxis().CenterTitle(True)
    hist.GetYaxis().CenterTitle(True)
    hist.SetLineColor(rt.kBlue)

    # Draw
    hist.Draw("E")
    decayp.SetLineColor(rt.kRed)
    decayp.Draw("x>1 same")
    bckgrnd.Draw("same")
    fitgraph.Draw("x>1")
    hist.Print("V")
    # legend.Draw()  # Uncomment if desired

    fitgraph.SaveAs("Gamma_1157_smallrange_bckgrndsub.png")

In [125]:
gamma_canvas = rt.TCanvas("gamma_canvas", "gamma_1158")
gamma_1158 = dtime.ProjectionX("Decay_Curve_1158keV",1155,1160)

Warning in <TCanvas::Constructor>: Deleting canvas with same name: gamma_canvas


In [126]:
bckgrnd_1158 = dtime.ProjectionX("Decay_Curve_1158",1133,1138)
gamma_1158.Add(bckgrnd_1158, -1)
Decayp(gamma_1158)
gamma_1158.Draw()
gamma_canvas.Update()
gamma_canvas.Draw()

Min X:  0
Max X:  1000.0
Initial p0 = 600.0
Initial p1 = 0.005644521014331802
Initial p2 = 0.0012765141446776157
Initial p3 = 0.3
Initial p4 = 400.0


Parameter 0 (source #) limits: [c_double(400.0), c_double(2000.0)]
Parameter 1 (Source lambda) limits: [c_double(0.005509913994912125), c_double(0.005785869620700712)]
Parameter 2 (Daughter lambda) limits: [c_double(1e-06), c_double(0.004)]
Parameter 3 (scaling factor) limits: [c_double(0.01), c_double(1.0)]
Parameter 4 (shift) limits: [c_double(0.0), c_double(800.0)]
Decay: 119.800003 ms ± 2.947763 ms
Decay Daughter: 173.286830 ms ± 25.506980 ms
Chi^2/NDF: 1.261336

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      1255.03
NDf                       =          995
Edm                       =  2.77478e-06
NCalls                    =          500
source #                  =      550.681   +/-   215.154      	 (limited)
Source lambda             =   0.00578587   +/-   0.000142365  	 (lim

Info in <TCanvas::Print>: png file Gamma_1157_smallrange_bckgrndsub.png has been created
